In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd, re, glob, math
from google.colab import files


TAHUN_MIN               = 2019
WILAYAH_SEKITAR         = False
BUANG_LAPORAN_PERBAIKAN = True
BUANG_TROTOAR           = True
BUANG_JPO               = True
BUANG_TOL               = True
WAJIB_LOKASI_SPESIFIK   = False
MARGIN_ERROR            = 0.05

csv_files = [f for f in glob.glob('*.csv')
             if not f.startswith(('hasil_', 'data_final', 'terbuang_', 'sampel_'))]
print("File terbaca:", csv_files)
if not csv_files:
    raise SystemExit("Tidak ada CSV. Jalankan ulang sel upload.")

df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
log = [("1. Total mentah", len(df))]

df = df.drop_duplicates(subset='id_str')
log.append(("2. Dedup ID", len(df)))

df = df[~df['full_text'].astype(str).str.strip().str.startswith('RT @')]
log.append(("3. Buang retweet", len(df)))

_n = df['full_text'].map(lambda t: re.sub(r'\s+', ' ',
     re.sub(r'[^a-z0-9\s]', ' ', re.sub(r'https?://\S+', '', str(t).lower()))).strip())
df = df[~_n.duplicated()]
log.append(("4. Buang teks kembar", len(df)))

df['tahun'] = pd.to_datetime(df['created_at'], format='mixed', utc=True).dt.year
if TAHUN_MIN:
    df = df[df['tahun'] >= TAHUN_MIN]
    log.append((f"5. Tahun >= {TAHUN_MIN}", len(df)))


WIL = {
 'Jakarta Pusat':  ['jakarta pusat','jakpus','menteng','gambir','senen','cempaka putih','johar baru','kemayoran','sawah besar','tanah abang','medan merdeka','gunung sahari','thamrin','monas','pasar baru','salemba','kwitang','kramat raya'],
 'Jakarta Utara':  ['jakarta utara','jakut','tanjung priok','koja','cilincing','penjaringan','pademangan','kelapa gading','sunter','ancol','marunda','enggano','yos sudarso','boulevard raya','gedong panjang','plumpang'],
 'Jakarta Barat':  ['jakarta barat','jakbar','cengkareng','kalideres','grogol','tambora','taman sari','kebon jeruk','palmerah','kembangan','tomang','slipi','glodok','s parman','daan mogot','kedoya','meruya','joglo','semanan','roxy'],
 'Jakarta Selatan':['jakarta selatan','jaksel','kebayoran','tebet','mampang','pancoran','jagakarsa','pasar minggu','cilandak','pesanggrahan','setiabudi','kemang','manggarai','lenteng agung','rawajati','kuningan','blok m','fatmawati','antasari','gatot subroto','gatsu','simatupang','radio dalam','senopati','casablanca','warung buncit','ragunan','pondok indah','rasuna said','sudirman'],
 'Jakarta Timur':  ['jakarta timur','jaktim','matraman','jatinegara','cakung','duren sawit','kramat jati','makasar','pulo gadung','pulogadung','ciracas','cipayung','pasar rebo','otista','pramuka','rawamangun','klender','cawang','kebon nanas','pondok bambu','halim','cibubur','penggilingan','buaran','pulo gebang','radjiman','kalimalang'],
 'Kep. Seribu':    ['kepulauan seribu','pulau seribu'],
}
SEKITAR = ['bekasi','tangerang','depok','bogor','tangsel','cikarang','bintaro','ciputat']
JAUH = ['bandung','cirebon','karawang','cikampek','sukabumi','lampung','surabaya','semarang','medan',
        'jawa tengah','bitung','balaraja','antapani','pantura','japek','bocimi','merak','sumatra',
        'kaltim','papua','aceh','kalimantan','jogja','solo','bali','ntt','flores']

def wilayah(t):
    t = ' ' + str(t).lower() + ' '
    hit = [w for w, kw in WIL.items() if any(k in t for k in kw)]
    if len(hit) == 1: return hit[0]
    if len(hit) > 1:  return 'AMBIGU: ' + ' | '.join(hit)
    return ''

def cari(t, daftar):
    t = str(t).lower()
    return ', '.join(sorted({k for k in daftar if k in t}))

STOP = (r'(?:depan|deket|dekat|arah|sebelah|menuju|sampai|tepatnya|tepat|yang|yg|udah|sudah|tolong|mohon|'
        r'banget|bgt|parah|rusak|bolong|berlubang|hancur|ambles|tuh|nih|dong|dan|cc|via|kondisi|akibat|'
        r'dihimbau|himbau|pengguna|tersebut|agar|kembali|penghubung|tampak|mulai|akan|bisa|sering|kerap|'
        r'banyak|hingga|serta|selesai|telah|oleh|untuk|pada|dari|masih|juga|hati)\b')
BUANG_KATA = {'raya','ini','itu','di','ke','jakarta'}

def jalan(t):
    t = re.sub(r'https?://\S+', '', str(t)); out = []
    for c in re.findall(r'(?:jl\.?|jln\.?|jalan)\s+([A-Za-z0-9\.\-\s]{3,45})', t, flags=re.I):
        c = re.split(STOP, c, flags=re.I)[0]
        c = re.sub(r'\s+', ' ', c).strip(' .-,')
        if len(c) >= 3 and c.lower() not in BUANG_KATA:
            out.append(c.title())
    return ' ; '.join(dict.fromkeys(out))

tx = df['full_text'].astype(str)
df['wilayah']    = tx.map(wilayah)
df['sekitar']    = tx.map(lambda t: cari(t, SEKITAR))
df['luar_jkt']   = tx.map(lambda t: cari(t, JAUH))
df['nama_jalan'] = tx.map(jalan)

df['status_lokasi'] = 'tanpa lokasi'
df.loc[tx.str.lower().str.contains('jakarta|jkt|dki', na=False), 'status_lokasi'] = 'Jakarta (umum)'
df.loc[df['sekitar']  != '', 'status_lokasi'] = 'sekitar Jakarta'
df.loc[df['luar_jkt'] != '', 'status_lokasi'] = 'luar Jabodetabek'
df.loc[df['wilayah'].str.startswith('AMBIGU'), 'status_lokasi'] = 'ambigu'
df.loc[df['wilayah'].str.startswith(('Jakarta','Kep')), 'status_lokasi'] = 'DKI'

simpan = ['DKI','Jakarta (umum)','ambigu'] + (['sekitar Jakarta'] if WILAYAH_SEKITAR else [])
df = df[df['status_lokasi'].isin(simpan)]
log.append(("6. Area Jakarta", len(df)))


JALAN = r'(?:jalanan|jalan|jln\.?|jl\.?|aspal|ruas|badan jalan|beton|cor)'
RUSAK = r'(?:rusak|berlubang|bolong|lubang|ambles|amblas|retak|bergelombang|menganga|keriting|gronjal|hancur)'
OBJ   = (r'(?:jpo|jembatan penyeberangan|jembatan penyebrangan|jembatan|cctv|kamera|sekolah|gedung|warteg|'
         r'kios|lapak|gardu|halte|kabel|pagar|atap|plafon|rumah|ruko|bus|truk|mobil|motor|kereta|krl|kapal|'
         r'saluran|got|trotoar|kanstin|barrier|lampu|pju|tiang|billboard|baliho|spanduk|taman|pot|median|'
         r'tanaman|ac|mesin|ban|kendaraan|hp|laptop|kamar|kantor|pabrik|rem|kopling|aki)')

LOK = re.compile(r'(?:\bdi\b|\bpada\b|\bsepanjang\b|\bkawasan\b|\bdepan\b|\bseberang\b|'
                 r'\bsekitar\b|\bdekat\b|\barah\b)\s*$', re.I)
re_rusak = re.compile(rf'\b{RUSAK}\b', re.I)
re_jalan = re.compile(rf'\b{JALAN}\b', re.I)
re_obj   = re.compile(rf'\b{OBJ}\b', re.I)
FWD      = re.compile(rf'^[ ,]*(?:di|pada|pd)?\s*(?:ruas\s+)?{JALAN}\b', re.I)
IDIOM    = re.compile(r'\b(?:siang bolong|tembok bolong|gali lubang|tutup lubang|lubang berjalan|jalan kaki|'
                      r'jalan-jalan|jalan2|turun ke jalan|jalan hidup|jalan cerita|jalan keluar|jalan tengah|'
                      r'jalan pintas|jalan buntu|jalan damai|jalan panjang|jalan terus|jalan pikiran|'
                      r'setengah jalan|jalan sehat|jalan santai|sepanjang jalan)\b', re.I)

def subjek_jalan(txt):
    t = re.sub(r'https?://\S+', ' ', str(txt))
    if IDIOM.search(t):
        return False
    for m in re_rusak.finditer(t):
        if FWD.match(t[m.end():m.end()+40]):
            return True
        win = re.split(r'[.!?\n]', t[max(0, m.start()-90):m.start()])[-1]
        j = list(re_jalan.finditer(win)); o = list(re_obj.finditer(win))
        if not j:
            continue
        lj = j[-1]; lo = o[-1] if o else None
        j_lok = bool(LOK.search(win[:lj.start()]))
        if lo is None:
            if not j_lok: return True
            continue
        o_lok = bool(LOK.search(win[:lo.start()]))
        if o_lok and not j_lok: return True
        if lo.start() > lj.start() or j_lok: continue
        return True
    return False

df['subjek_jalan'] = df['full_text'].astype(str).map(subjek_jalan)
df[~df['subjek_jalan']].to_csv('terbuang_topik.csv', index=False)
df = df[df['subjek_jalan']]
log.append(("7. Subjek = jalan", len(df)))


tx2 = df['full_text'].astype(str)
PERB = re.compile(r'\b(?:diperbaiki|perbaikan|memperbaiki|perbaiki|ditambal|penambalan|menambal|diaspal|'
                  r'pengaspalan|rampung|selesai dikerjakan|pasukan kuning|satgas|coldmix|sudah diperbaiki|'
                  r'dikerjakan|pemeliharaan rutin|sudin bina marga|suku dinas bina marga|sdbm|'
                  r'pengaturan dan pengamanan)\b', re.I)
KELUH = re.compile(r'\b(?:tolong|mohon|keluhan|dikeluhkan|mengeluh|kapan|belum juga|tak kunjung|tidak kunjung|'
                   r'dibenerin|segera|minta|rawan|bahaya|membahayakan|hati-?hati|hati2|waspada|korban|tewas|'
                   r'meninggal|kecelakaan|please)\b', re.I)

df['jenis'] = 'keluhan'
df.loc[tx2.str.contains(PERB) & ~tx2.str.contains(KELUH), 'jenis'] = 'laporan perbaikan'
df['trotoar'] = tx2.str.contains(r'\btrotoar\b', case=False)
df['jpo']     = tx2.str.contains(r'\b(?:jpo|jembatan penyeberangan|jembatan penyebrangan|flyover|fly over|jembatan layang)\b', case=False, regex=True)
df['tol']     = tx2.str.contains(r'\b(?:tol|jorr|japek|wiyoto)\b', case=False, regex=True)
df['lokasi_spesifik'] = (df['wilayah'] != '') | (df['nama_jalan'] != '')

if BUANG_LAPORAN_PERBAIKAN:
    df = df[df['jenis'] == 'keluhan'];   log.append(("8. Buang laporan perbaikan", len(df)))
if BUANG_TROTOAR:
    df = df[~df['trotoar']];             log.append(("9. Buang trotoar", len(df)))
if BUANG_JPO:
    df = df[~df['jpo']];                 log.append(("10. Buang JPO/flyover", len(df)))
if BUANG_TOL:
    df = df[~df['tol']];                 log.append(("11. Buang tol", len(df)))
if WAJIB_LOKASI_SPESIFIK:
    df = df[df['lokasi_spesifik']];      log.append(("12. Wajib lokasi spesifik", len(df)))


print("\n=== ALUR PEMBERSIHAN ===")
for nama, jml in log:
    print(f"{nama:<32}: {jml}")

N = len(df)
n = math.ceil(N / (1 + N * MARGIN_ERROR**2))
print(f"\n=== DATA FINAL: {N} ===")
print(f"Slovin (e={MARGIN_ERROR:.0%}) -> sampel anotasi manual: {n}")
print("\n--- Wilayah ---")
print(df['wilayah'].replace('', '(tanpa wilayah)').value_counts().to_string())
print("\n--- Tahun ---")
print(df['tahun'].value_counts().sort_index().to_string())

kol = [c for c in ['id_str','created_at','tahun','username','lang','full_text',
                   'wilayah','nama_jalan','status_lokasi','jenis','tweet_url'] if c in df.columns]
df[kol].to_csv('data_final.csv', index=False)

sampel = df[kol].sample(n=min(n, N), random_state=42).copy()
sampel['relevan'] = ''
sampel['urgensi'] = ''
sampel.to_csv('sampel_label_manual.csv', index=False)

print(f"\nTersimpan: data_final.csv ({N}) | sampel_label_manual.csv ({len(sampel)})")
files.download('data_final.csv')
files.download('sampel_label_manual.csv')